In [9]:
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import pickle
%matplotlib inline

In [10]:
INPUT_PATH = "data/facsimile_data.csv"
OUTPUT_MODEL_READY_PATH = "data/facsimile_model_ready_data.csv"
OUTPUT_DEMOGRAPHICS_PATH = "data/demographics.csv"

RNN_data = pd.read_csv(INPUT_PATH)

if "Unnamed: 0" in RNN_data.columns:
    RNN_data = RNN_data.drop(columns=["Unnamed: 0"])

rename_map = {}

if "Age.FirstDose" in RNN_data.columns:
    rename_map["Age.FirstDose"] = "Age"
if "severe_infection_next" in RNN_data.columns:
    rename_map["severe_infection_next"] = "sev_inf_next"
if "Visits" in RNN_data.columns:
    rename_map["Visits"] = "Visits"
if "windex" in RNN_data.columns:
    rename_map["windex"] = "windex"

RNN_data = RNN_data.rename(columns=rename_map)

required_cols = [
    "id", "action", "Age", "Gender", "Race", "Visits",
    "imm_baseline", "windex", "numVax", "variant",
    "sev_inf_next", "inf_next"
]
missing = [c for c in required_cols if c not in RNN_data.columns]
if missing:
    raise ValueError(f"Missing required columns in facsimile_data.csv: {missing}")

RNN_data.loc[RNN_data["Age"] == ">89", "Age"] = 90
RNN_data["Age"] = pd.to_numeric(RNN_data["Age"], errors="raise").astype(int)

RNN_data = RNN_data.sort_values(["id"]).reset_index(drop=True)
RNN_data["month_index"] = RNN_data.groupby("id").cumcount()

where_severe_inf = np.where(RNN_data["sev_inf_next"].values == 1)[0]
RNN_data.loc[where_severe_inf, "inf_next"] = 1

RNN_data.head()

,id,action,Age,Gender,Race,Visits,imm_baseline,windex,numVax,variant,sev_inf_next,inf_next,month_index
0,1,0,51,F,Other,18,0,2,0,none,0,0,0
1,1,0,51,F,Other,18,0,2,0,none,0,0,1
2,1,0,51,F,Other,18,0,2,0,none,0,0,2
3,1,0,51,F,Other,18,0,2,0,none,0,0,3
4,1,0,51,F,Other,18,0,2,0,none,0,0,4


In [11]:
scaler = StandardScaler()

RNN_data_demographics = RNN_data.drop_duplicates(subset=["id"]).copy()

age_raw = RNN_data_demographics[["Age"]].astype(int)
normalized_age = scaler.fit_transform(age_raw)
age_map = dict(zip(RNN_data_demographics["Age"], normalized_age.reshape(-1)))

RNN_data["Age_standardized"] = RNN_data["Age"].map(age_map)

RNN_data["age_cat"] = pd.cut(
    RNN_data["Age"],
    bins=[0, 18, 30, 50, 65, 100],
    include_lowest=True,
    right=False,
    labels=False,
).astype(int)

RNN_data[["id", "Age", "Age_standardized", "age_cat"]].head()

,id,Age,Age_standardized,age_cat
0,1,51,0.192017,3
1,1,51,0.192017,3
2,1,51,0.192017,3
3,1,51,0.192017,3
4,1,51,0.192017,3


In [12]:
def compute_months_since_vax(group: pd.DataFrame) -> pd.DataFrame:
    group = group.sort_values("month_index").copy()
    months_since = []
    last_vax_idx = None

    for _, row in group.iterrows():
        if last_vax_idx is None:
            cur = 999
        else:
            cur = int(row["month_index"] - last_vax_idx)

        months_since.append(cur)

        if int(row["action"]) == 1:
            last_vax_idx = int(row["month_index"])

    group["months_since_vax"] = months_since
    return group

pieces = []
for pid, g in RNN_data.groupby("id", sort=False):
    pieces.append(compute_months_since_vax(g))

RNN_data = pd.concat(pieces, axis=0).reset_index(drop=True)

RNN_data["months_since_vax_cat"] = pd.cut(
    RNN_data["months_since_vax"],
    bins=[0, 5, 7, 1000],
    include_lowest=True,
    right=False,
    labels=False,
)

RNN_data["months_since_vax_cat"] = RNN_data["months_since_vax_cat"].astype(int)

print(RNN_data.columns.tolist())
RNN_data[["id", "month_index", "action", "months_since_vax", "months_since_vax_cat"]].head(15)

['id', 'action', 'Age', 'Gender', 'Race', 'Visits', 'imm_baseline', 'windex', 'numVax', 'variant', 'sev_inf_next', 'inf_next', 'month_index', 'Age_standardized', 'age_cat', 'months_since_vax', 'months_since_vax_cat']


,id,month_index,action,months_since_vax,months_since_vax_cat
0,1,0,0,999,2
1,1,1,0,999,2
2,1,2,0,999,2
3,1,3,0,999,2
4,1,4,0,999,2
5,1,5,0,999,2
6,1,6,0,999,2
7,1,7,0,999,2
8,1,8,0,999,2
9,1,9,0,999,2


In [13]:
race_dummies = pd.get_dummies(RNN_data["Race"], prefix="race")
gender_male = (RNN_data["Gender"] == "M").astype(int).rename("gender_male")
variant_dummies = pd.get_dummies(RNN_data["variant"], prefix="variant")

visits_dummies = pd.get_dummies(
    pd.cut(
        RNN_data["Visits"],
        bins=[0, 5, 10, 20, 50, 1000],
        include_lowest=True,
        right=False
    ),
    prefix="visits"
)

windex_dummies = pd.get_dummies(
    pd.cut(
        RNN_data["windex"],
        bins=[0, 1, 3, 5, 100],
        include_lowest=True,
        right=False
    ),
    prefix="windex"
)

age_cat_dummies = pd.get_dummies(RNN_data["age_cat"], prefix="agecat")

race_dummies.head()

,race_African American,race_Caucasian,race_Other
0,False,False,True
1,False,False,True
2,False,False,True
3,False,False,True
4,False,False,True


In [14]:
model_ready = pd.concat(
    [
        RNN_data[
            [
                "id",
                "month_index",
                "action",
                "Age",
                "Age_standardized",
                "age_cat",
                "imm_baseline",
                "months_since_vax",
                "months_since_vax_cat",
                "numVax",
                "Visits",
                "windex",
                "variant",
                "sev_inf_next",
                "inf_next",
            ]
        ],
        gender_male,
        race_dummies,
        visits_dummies,
        windex_dummies,
        variant_dummies,
        age_cat_dummies,
    ],
    axis=1,
)

model_ready.head()

,id,month_index,action,Age,Age_standardized,age_cat,imm_baseline,months_since_vax,months_since_vax_cat,numVax,...,"windex_[3, 5)","windex_[5, 100)",variant_delta,variant_none,variant_omicron,agecat_0,agecat_1,agecat_2,agecat_3,agecat_4
0,1,0,0,51,0.192017,3,0,999,2,0,...,False,False,False,True,False,False,False,False,True,False
1,1,1,0,51,0.192017,3,0,999,2,0,...,False,False,False,True,False,False,False,False,True,False
2,1,2,0,51,0.192017,3,0,999,2,0,...,False,False,False,True,False,False,False,False,True,False
3,1,3,0,51,0.192017,3,0,999,2,0,...,False,False,False,True,False,False,False,False,True,False
4,1,4,0,51,0.192017,3,0,999,2,0,...,False,False,False,True,False,False,False,False,True,False


In [15]:
print(model_ready.columns.tolist())
print(model_ready.shape)
model_ready.head()

['id', 'month_index', 'action', 'Age', 'Age_standardized', 'age_cat', 'imm_baseline', 'months_since_vax', 'months_since_vax_cat', 'numVax', 'Visits', 'windex', 'variant', 'sev_inf_next', 'inf_next', 'gender_male', 'race_African American', 'race_Caucasian', 'race_Other', 'visits_[0, 5)', 'visits_[5, 10)', 'visits_[10, 20)', 'visits_[20, 50)', 'visits_[50, 1000)', 'windex_[0, 1)', 'windex_[1, 3)', 'windex_[3, 5)', 'windex_[5, 100)', 'variant_delta', 'variant_none', 'variant_omicron', 'agecat_0', 'agecat_1', 'agecat_2', 'agecat_3', 'agecat_4']
(26378, 36)


,id,month_index,action,Age,Age_standardized,age_cat,imm_baseline,months_since_vax,months_since_vax_cat,numVax,...,"windex_[3, 5)","windex_[5, 100)",variant_delta,variant_none,variant_omicron,agecat_0,agecat_1,agecat_2,agecat_3,agecat_4
0,1,0,0,51,0.192017,3,0,999,2,0,...,False,False,False,True,False,False,False,False,True,False
1,1,1,0,51,0.192017,3,0,999,2,0,...,False,False,False,True,False,False,False,False,True,False
2,1,2,0,51,0.192017,3,0,999,2,0,...,False,False,False,True,False,False,False,False,True,False
3,1,3,0,51,0.192017,3,0,999,2,0,...,False,False,False,True,False,False,False,False,True,False
4,1,4,0,51,0.192017,3,0,999,2,0,...,False,False,False,True,False,False,False,False,True,False


In [16]:
model_ready.to_csv(OUTPUT_MODEL_READY_PATH, index=False)

demographics = (
    model_ready
    .drop_duplicates(subset=["id"])
    .drop(
        columns=[
            "month_index",
            "action",
            "numVax",
            "months_since_vax",
            "months_since_vax_cat",
            "variant",
            "sev_inf_next",
            "inf_next",
        ],
        errors="ignore",
    )
)

demographics.to_csv(OUTPUT_DEMOGRAPHICS_PATH, index=False)

print("saved:", OUTPUT_MODEL_READY_PATH)
print("saved:", OUTPUT_DEMOGRAPHICS_PATH)

saved: data/facsimile_model_ready_data.csv
saved: data/demographics.csv


In [ ]:
rnn_covariate_cols = [
    "action",
    "Age_standardized",
    "imm_baseline",
    "numVax",
    "gender_male",
]

for col in race_dummies.columns:
    if col != "race_Caucasian":
        rnn_covariate_cols.append(col)

for col in visits_dummies.columns[1:]:
    rnn_covariate_cols.append(col)

for col in windex_dummies.columns[1:]:
    rnn_covariate_cols.append(col)

for col in variant_dummies.columns:
    if col != "variant_none":
        rnn_covariate_cols.append(col)

rnn_outcome_cols = ["sev_inf_next", "inf_next"]

n = model_ready["id"].nunique()
p = len(rnn_covariate_cols)
t = model_ready.groupby("id").size().max()
q = len(rnn_outcome_cols)

covariates_rnn = np.zeros((n, t, p), dtype=np.float32)
outcomes_rnn = np.zeros((n, t, q), dtype=np.float32)
seq_length = np.zeros(n, dtype=np.int64)

print("n =", n, "p =", p, "t =", t, "q =", q)
print(rnn_covariate_cols)

n = 1000 p = 16 t = 27 q = 2
['action', 'Age_standardized', 'imm_baseline', 'numVax', 'gender_male', 'race_African American', 'race_Other', 'visits_[5, 10)', 'visits_[10, 20)', 'visits_[20, 50)', 'visits_[50, 1000)', 'windex_[1, 3)', 'windex_[3, 5)', 'windex_[5, 100)', 'variant_delta', 'variant_omicron']


In [18]:
grouped_data = model_ready.groupby("id", sort=False)

for i, (pt_id, group) in enumerate(grouped_data):
    group = group.sort_values("month_index")

    covariates_i = group[rnn_covariate_cols].to_numpy(dtype=np.float32)
    outcomes_i = group[rnn_outcome_cols].to_numpy(dtype=np.float32)

    time_i = np.arange(group.shape[0])

    severe_pos = np.where(outcomes_i[:, 0] == 1)[0]
    if len(severe_pos) > 0:
        seq_length[i] = int(severe_pos[0] + 1)
    else:
        seq_length[i] = int(len(time_i))

    covariates_rnn[i, time_i, :] = covariates_i
    outcomes_rnn[i, time_i, :] = outcomes_i

    if (i + 1) % 100 == 0:
        print(f"{i + 1} / {n}")

100 / 1000
200 / 1000
300 / 1000
400 / 1000
500 / 1000
600 / 1000
700 / 1000
800 / 1000
900 / 1000
1000 / 1000


In [19]:
np.save("data/covariates_rnn.npy", covariates_rnn)
np.save("data/outcomes_rnn.npy", outcomes_rnn)
np.save("data/seq_length.npy", seq_length)

print("saved: data/covariates_rnn.npy")
print("saved: data/outcomes_rnn.npy")
print("saved: data/seq_length.npy")

saved: data/covariates_rnn.npy
saved: data/outcomes_rnn.npy
saved: data/seq_length.npy
